# TimesFM Best-of-N (Algorithm 3) fan chart tren ERA5

**Truoc khi chay:**
1. Add Data -> upload file `era5_test_2017_2018.nc` thanh 1 Kaggle Dataset, roi attach vao notebook nay.
2. Sua `DATA_PATH` o cell config cho dung duong dan Kaggle sinh ra (thuong la `/kaggle/input/<ten-dataset>/era5_test_2017_2018.nc`).
3. Bat GPU o Settings > Accelerator (khong bat buoc, nhung nhanh hon nhieu).
4. Chay lan luot cac cell ben duoi.

Script nay sinh N=20 candidate tuong lai tu TimesFM theo dung Algorithm 3 (Best-of-N Sample,
nhanh quantile-head: per-timestep inverse-CDF sampling tren luoi quantile co dinh [0.1..0.9]),
ve chung voi lich su va ground truth.

In [ ]:
!pip install -q timesfm xarray netCDF4

In [ ]:
# ==== CONFIG - sua lai cho dung moi truong Kaggle cua ban ====
DATA_PATH = "/kaggle/input/era5-test-2017-2018/era5_test_2017_2018.nc"
TARGET_VAR = "2m_temperature"
HISTORY_STEPS = 512
PREDICTION_STEPS = 64
MODEL_ID = "google/timesfm-2.5-200m-pytorch"
N_SAMPLES = 20
START_IDX = 0
LAT_IDX = 16
LON_IDX = 32
HISTORY_TAIL = 96
SEED = 0
OUT_PATH = "/kaggle/working/fan_chart_t2m_timesfm.png"

In [ ]:
import numpy as np
import torch
import xarray as xr
import matplotlib.pyplot as plt

"""
Algorithm 3 (Best-of-N Sample), nhanh quantile head (TimesFM khong co sampler
that - sampler="quantile"):

    Fix dense quantile grid G
    for i = 1..N-1:
        u_{i,t} ~ U(0,1) i.i.d. tung buoc thoi gian t
        y_i[t] = F_theta^{-1}(u_{i,t} | t, G)      # noi suy nghich dao CDF
    Luon them du bao median (muc 0.5) la 1 candidate trong N.
"""

DEFAULT_QUANTILE_LEVELS = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], dtype=np.float32)


def detect_version(model_id: str) -> str:
    m = model_id.lower()
    if "2.0" in m or "2p0" in m:
        return "2.0"
    if "3.0" in m or "3p0" in m or "timesfm-3" in m or "timesfm3" in m:
        return "3.0"
    return "2.5"


def find_new_style_class(timesfm_mod, version: str):
    tag = version.replace(".", "p")
    candidates = [
        n for n in dir(timesfm_mod)
        if "torch" in n.lower() and hasattr(getattr(timesfm_mod, n), "from_pretrained")
    ]
    for n in candidates:
        if tag in n.lower().replace("_", ""):
            return getattr(timesfm_mod, n), n
    if candidates:
        return getattr(timesfm_mod, candidates[0]), candidates[0]
    return None, None


def build_timesfm(model_id: str, device: str, history_steps: int, prediction_steps: int):
    import timesfm

    version = detect_version(model_id)
    api = "old" if version == "2.0" else "new"
    print(f"[TimesFM] model_id={model_id} version={version} api={api} device={device}")

    if api == "old":
        backend = "gpu" if device == "cuda" else "cpu"
        hparams = timesfm.TimesFmHparams(
            backend=backend,
            per_core_batch_size=32,
            horizon_len=max(prediction_steps, 128),
            context_len=min(max(history_steps, 512), 2048),
            num_layers=50,
            use_positional_embedding=False,
        )
        ckpt = timesfm.TimesFmCheckpoint(huggingface_repo_id=model_id)
        model = timesfm.TimesFm(hparams=hparams, checkpoint=ckpt)
        return model, api, version

    cls, cls_name = find_new_style_class(timesfm, version)
    if cls is None:
        available = [n for n in dir(timesfm) if "torch" in n.lower()]
        raise RuntimeError(
            f"Khong tim thay class loader new-style cho TimesFM {version}. "
            f"Cac ten *_torch co san: {available}."
        )
    print(f"[TimesFM] new-style class: timesfm.{cls_name}")
    model = cls.from_pretrained(model_id)
    try:
        model.compile(
            timesfm.ForecastConfig(
                max_context=min(max(history_steps, 512), 2048),
                max_horizon=max(prediction_steps, 256),
                normalize_inputs=True,
                use_continuous_quantile_head=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            )
        )
    except TypeError as e:
        print(f"[TimesFM] .compile() khong ap dung cho {cls_name} ({e}); bo qua.")
    return model, api, version


def forecast_point_and_quantiles(model, api: str, history_1d: np.ndarray, prediction_steps: int):
    inputs = [history_1d.astype(np.float32)]
    if api == "old":
        point, q = model.forecast(inputs, freq=[0])
        q = np.asarray(q)[0, :prediction_steps, :]
    else:
        point, q = model.forecast(horizon=prediction_steps, inputs=inputs)
        q = np.asarray(q)[0, :prediction_steps, :]
        if q.shape[-1] == 10:
            q = q[:, 1:]  # bo cot 0 (mean), giu 9 cot quantile 0.1..0.9

    if q.shape[-1] != len(DEFAULT_QUANTILE_LEVELS):
        raise RuntimeError(
            f"So cot quantile tra ve ({q.shape[-1]}) khac gia dinh ({len(DEFAULT_QUANTILE_LEVELS)})."
        )

    point = np.asarray(point, dtype=np.float32)[0, :prediction_steps]
    return point, q.astype(np.float32), DEFAULT_QUANTILE_LEVELS


def sample_best_of_n(quantile_forecast, quantile_levels, n_samples, rng):
    H, _Q = quantile_forecast.shape
    n_random = max(n_samples - 1, 0)

    u = rng.uniform(0.0, 1.0, size=(n_random, H)).astype(np.float32)
    samples = np.empty((n_random, H), dtype=np.float32)
    for t in range(H):
        samples[:, t] = np.interp(u[:, t], quantile_levels, quantile_forecast[t, :])

    median = np.empty((H,), dtype=np.float32)
    for t in range(H):
        median[t] = np.interp(0.5, quantile_levels, quantile_forecast[t, :])

    return np.concatenate([samples, median[np.newaxis, :]], axis=0)

In [ ]:
rng = np.random.default_rng(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

ds = xr.open_dataset(DATA_PATH).transpose("time", "latitude", "longitude")
h_start, h_end = START_IDX, START_IDX + HISTORY_STEPS
g_end = h_end + PREDICTION_STEPS

series_full = ds[TARGET_VAR].values  # (time, lat, lon)
history_1d = series_full[h_start:h_end, LAT_IDX, LON_IDX]
gt_1d = series_full[h_end:g_end, LAT_IDX, LON_IDX]

model, api, version = build_timesfm(MODEL_ID, device, HISTORY_STEPS, PREDICTION_STEPS)
point, quantile_forecast, quantile_levels = forecast_point_and_quantiles(
    model, api, history_1d, PREDICTION_STEPS
)
candidates = sample_best_of_n(quantile_forecast, quantile_levels, N_SAMPLES, rng)  # (N, H)
print("candidates shape:", candidates.shape)

In [ ]:
history_tail_1d = history_1d[-HISTORY_TAIL:]
history_x = np.arange(-len(history_tail_1d), 0)
future_x = np.arange(0, PREDICTION_STEPS)

plt.figure(figsize=(11, 5))
plt.plot(history_x, history_tail_1d, color="black", linewidth=1.8, label="Lich su (history)")
for i in range(N_SAMPLES):
    plt.plot(future_x, candidates[i], color="tab:green", alpha=0.25, linewidth=1.0)
plt.plot(future_x, candidates.mean(axis=0), color="tab:green", linewidth=2.0,
          label=f"Trung binh {N_SAMPLES} candidate (TimesFM, quantile-sampling)")
plt.plot(future_x, point, color="tab:orange", linewidth=1.5, linestyle="-.",
          label="Point forecast (TimesFM, tat dinh)")
plt.plot(future_x, gt_1d, color="tab:red", linewidth=1.8, linestyle="--", label="Ground truth")
plt.axvline(0, color="gray", linestyle=":", linewidth=1.0)
plt.xlabel("Buoc thoi gian (0 = thoi diem hien tai)")
plt.ylabel(f"{TARGET_VAR} (K)")
plt.title(
    f"TimesFM ({version}): 1 lich su -> {N_SAMPLES} candidate "
    f"(Algorithm 3, quantile-head sampling) - lat={LAT_IDX}, lon={LON_IDX}"
)
plt.legend()
plt.tight_layout()
plt.savefig(OUT_PATH, dpi=150)
plt.show()
print(f"Da luu hinh -> {OUT_PATH}")